<a href="https://colab.research.google.com/github/rcsb/rcsb-training-resources/blob/master/training-events/2026/python-rcsb-api/sequence_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using `rcsb-api` to access RCSB PDB's Sequence Coordinates API

In [ ]:
# Install `rcsb-api`
%pip install --upgrade rcsb-api

# Install `rich` for pretty printing
%pip install rich

## Sequence Coordinates API Query Basics
The Sequence Coordinates API ([sequence-coordinates.rcsb.org](https://sequence-coordinates.rcsb.org/)) can be used to obtain alignments between various sequence databases as well as annotations and positional features for PDB sequences. There are two basic types of queries you can perform with the sequence coordinates API—`Alignments` and `Annotations`. 

- `Alignments` allows you to get sequence alignments and coverage between a PDB entity or instance, UniProt sequence, and/or NCBI genome or protein sequence.
- `Annotations` allows you to get positional features/annotations for a given PDB entity or instance, UniProt sequence, or NCBI genome or protein sequence. (Annotations are sourced from PDB data, UniProt, CATH, ECOD, SCOPe, and other third-party resources integrated into RCSB PDB data.)

### Arguments
Each type of query requires a similar but slightly different set of arguments. For the complete list of possible arguments for both types of Sequence API queries, see the [*rcsb-api* documentation](https://rcsbapi.readthedocs.io/en/latest/seq_api/quickstart.html#getting-started).

#### Alignments
- **Query sequence ID** (`query_id`): identifier of the PDB entity or instance, UniProt sequence, or NCBI genome or protein to align (following the format of the "Examples" in [SequenceReference table](#Table-of-possible-SequenceReference-values))
- **Query sequence type** (`db_from`):  sequence data source of the queried ID (see [SequenceReference table](#Table-of-possible-SequenceReference-values))
- **Target alignment** (`db_to`): target sequence dataset to align the sequence  (see [SequenceReference table](#Table-of-possible-SequenceReference-values))
- **Alignment data fields to return** (`return_data_list`): e.g., alignment positions, coverage, etc.

#### Annotations
- **Query sequence ID** (`query_id`): identifier of the PDB entity or instance, UniProt sequence, or NCBI genome or protein to align (following the format of the "Examples" in [SequenceReference table](#Table-of-possible-SequenceReference-values))
- **Query sequence type** (`reference`):  sequence dataset of the queried ID  (see [SequenceReference table](#Table-of-possible-SequenceReference-values))
- **Annotation sources** (`sources`): data sources to extract annotations from (one or more of [`UNIPROT`, `PDB_ENTITY`, `PDB_INSTANCE`])
- **Annotation data fields to return** (`return_data_list`): e.g., feature positions, name, values, etc.

#### Table of possible `SequenceReference` values
Both `Alignments` and `Annotations` queries make use of the same set of possible values to provide to sequence data source arguments. The table below describes the type of database identifiers used for each `SequenceReference` value.

| `SequenceReference` | Database Identifier Description              | Example                        |
|---------------------|-----------------------------------------------|--------------------------------|
| `NCBI_GENOME`       | NCBI RefSeq Chromosome Accession              | `NC_000001`                    |
| `NCBI_PROTEIN`      | NCBI RefSeq Protein Accession                 | `NP_789765`                    |
| `UNIPROT`           | UniProt Accession                             | `P01112`                       |
| `PDB_ENTITY`        | RCSB PDB Entity Id / CSM Entity Id            | `2UZI_3` / `AF_AFP68871F1_1`   |
| `PDB_INSTANCE`      | RCSB PDB Instance Id / CSM Instance Id        | `2UZI.C` / `AF_AFP68871F1.A`   |



## Creating a Sequence Alignment API Query

We'll start by creating a Sequence Alignment API query to determine the residue positions and coverage of an alignment.

### 1. Fetch the sequence alignment between PDB entity `3V85_1` and its corresponding UniProt sequence
([Protein sequence feature viewer](https://www.rcsb.org/sequence/3V85); <a href="https://sequence-coordinates.rcsb.org/graphiql/index.html?query=%7B%0A%20%20alignments(from%3A%20PDB_ENTITY%2C%20queryId%3A%20%223V85_1%22%2C%20to%3A%20UNIPROT)%20%7B%0A%20%20%20%20query_sequence%0A%20%20%20%20target_alignments%20%7B%0A%20%20%20%20%20%20target_sequence%0A%20%20%20%20%20%20orientation%0A%20%20%20%20%20%20aligned_regions%20%7B%0A%20%20%20%20%20%20%20%20target_end%0A%20%20%20%20%20%20%20%20query_begin%0A%20%20%20%20%20%20%20%20exon_shift%0A%20%20%20%20%20%20%20%20query_end%0A%20%20%20%20%20%20%20%20target_begin%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%20%20target_id%0A%20%20%20%20%20%20coverage%20%7B%0A%20%20%20%20%20%20%20%20query_length%0A%20%20%20%20%20%20%20%20target_length%0A%20%20%20%20%20%20%20%20target_coverage%0A%20%20%20%20%20%20%20%20query_coverage%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%7D%0A%20%20%20%20alignment_length%0A%20%20%7D%0A%7D">API Query Editor link</a>)

In [ ]:
from rcsbapi.sequence import Alignments
from rich import print as rprint

query = Alignments(
    query_id="3V85_1",
    db_from="PDB_ENTITY",
    db_to="UNIPROT",
    return_data_list=["query_sequence", "target_alignments", "alignment_length"]  # This specifies what data items to return
)
result_dict = query.exec()
rprint(result_dict)
print(f"API Query Editor link: {query.get_editor_link()}")

### 2. Fetch the alignment between PDB entity `3V85_1` and its corresponding UniProt sequence
Now let's get the alignment between the same entity and its corresponding NCBI Genome sequence ([Genome Viewer](https://www.rcsb.org/genome/3V85); <a href="https://sequence-coordinates.rcsb.org/graphiql/index.html?query=%7B%0A%20%20alignments(from%3A%20PDB_ENTITY%2C%20queryId%3A%20%223V85_1%22%2C%20to%3A%20NCBI_GENOME)%20%7B%0A%20%20%20%20query_sequence%0A%20%20%20%20target_alignments%20%7B%0A%20%20%20%20%20%20coverage%20%7B%0A%20%20%20%20%20%20%20%20query_coverage%0A%20%20%20%20%20%20%20%20target_coverage%0A%20%20%20%20%20%20%20%20target_length%0A%20%20%20%20%20%20%20%20query_length%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%20%20target_id%0A%20%20%20%20%20%20orientation%0A%20%20%20%20%20%20target_sequence%0A%20%20%20%20%20%20aligned_regions%20%7B%0A%20%20%20%20%20%20%20%20target_begin%0A%20%20%20%20%20%20%20%20exon_shift%0A%20%20%20%20%20%20%20%20target_end%0A%20%20%20%20%20%20%20%20query_begin%0A%20%20%20%20%20%20%20%20query_end%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%7D%0A%20%20%20%20alignment_length%0A%20%20%7D%0A%7D">API Query Editor link</a>):

In [ ]:
from rcsbapi.sequence import Alignments
from rich import print as rprint

query = Alignments(
    query_id="3V85.A",
    db_from="PDB_INSTANCE",
    db_to="UNIPROT",
    return_data_list=["query_sequence", "target_alignments", "alignment_length"]
)
result_dict = query.exec()
print(f"API Query Editor link: {query.get_editor_link()}")
rprint(result_dict)

### 3. Fetch the sequence alignment of *all* PDB entities of UniProt Q9SIY3
Last, rather than getting the alignment one entity at-a-time, we can request the sequence alignment of *all* PDB entities of UniProt Q9SIY3 ([UniProt Group Sequence Viewer](https://www.rcsb.org/groups/sequence/polymer_entity/Q9SIY3); <a href="https://sequence-coordinates.rcsb.org/graphiql/index.html?query=%7B%0A%20%20alignments(from%3A%20UNIPROT%2C%20queryId%3A%20%22Q9SIY3%22%2C%20to%3A%20PDB_ENTITY)%20%7B%0A%20%20%20%20query_sequence%0A%20%20%20%20target_alignments%20%7B%0A%20%20%20%20%20%20aligned_regions%20%7B%0A%20%20%20%20%20%20%20%20query_end%0A%20%20%20%20%20%20%20%20exon_shift%0A%20%20%20%20%20%20%20%20target_begin%0A%20%20%20%20%20%20%20%20query_begin%0A%20%20%20%20%20%20%20%20target_end%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%20%20target_id%0A%20%20%20%20%20%20coverage%20%7B%0A%20%20%20%20%20%20%20%20query_coverage%0A%20%20%20%20%20%20%20%20target_coverage%0A%20%20%20%20%20%20%20%20query_length%0A%20%20%20%20%20%20%20%20target_length%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%20%20target_sequence%0A%20%20%20%20%20%20orientation%0A%20%20%20%20%7D%0A%20%20%20%20alignment_length%0A%20%20%7D%0A%7D">API Query Editor link</a>):

In [ ]:
from rcsbapi.sequence import Alignments
from rich import print as rprint

query = Alignments(
    query_id="Q9SIY3",
    db_from="UNIPROT",
    db_to="PDB_ENTITY",
    return_data_list=["query_sequence", "target_alignments", "alignment_length"]
)
result_dict = query.exec()
print(f"API Query Editor link: {query.get_editor_link()}")
rprint(result_dict)

## Creating a Sequence Annotations API Query

Next, we'll look at creating a Sequence Annotations API query to fetch positional annotations associated to the queried sequence.

### 1. Fetch the sequence annotations for PDB instance 3V85.A
([Protein sequence feature viewer](https://www.rcsb.org/sequence/3V85); <a href="https://sequence-coordinates.rcsb.org/graphiql/index.html?query=%7B%0A%20%20annotations(%0A%20%20%20%20queryId%3A%20%223V85.A%22%0A%20%20%20%20reference%3A%20PDB_INSTANCE%0A%20%20%20%20sources%3A%20%5BUNIPROT%2C%20PDB_ENTITY%2C%20PDB_INSTANCE%5D%0A%20%20)%20%7B%0A%20%20%20%20target_id%0A%20%20%20%20source%0A%20%20%20%20features%20%7B%0A%20%20%20%20%20%20provenance_source%0A%20%20%20%20%20%20additional_properties%20%7B%0A%20%20%20%20%20%20%20%20property_name%0A%20%20%20%20%20%20%20%20property_value%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%20%20name%0A%20%20%20%20%20%20feature_positions%20%7B%0A%20%20%20%20%20%20%20%20value%0A%20%20%20%20%20%20%20%20open_begin%0A%20%20%20%20%20%20%20%20beg_ori_id%0A%20%20%20%20%20%20%20%20open_end%0A%20%20%20%20%20%20%20%20values%0A%20%20%20%20%20%20%20%20end_ori_id%0A%20%20%20%20%20%20%20%20range_id%0A%20%20%20%20%20%20%20%20beg_seq_id%0A%20%20%20%20%20%20%20%20end_seq_id%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%20%20description%0A%20%20%20%20%20%20value%0A%20%20%20%20%20%20feature_id%0A%20%20%20%20%20%20type%0A%20%20%20%20%7D%0A%20%20%7D%0A%7D">API Query Editor link</a>)

In [ ]:
from rcsbapi.sequence import Annotations

# Fetch all positional features for a particular PDB Instance
query = Annotations(
    query_id="3V85.A",
    reference="PDB_INSTANCE",
    sources=["UNIPROT", "PDB_ENTITY", "PDB_INSTANCE"],
    return_data_list=["target_id", "source", "features"]
)
result_dict = query.exec()
print("Number of feature/annotation sources: ", len(result_dict["data"]["annotations"]))
print(f"API Query Editor link: {query.get_editor_link()}")

In [ ]:
# Output of above is too large to print, so print out just the features/annotations from each source:
for d in result_dict["data"]["annotations"]:
    print(f"Number of features/annotations for source {d['source']}: {len(d['features'])}")
    fL = []
    if len(d["features"]) > 0:
        fL = [f["type"] for f in d["features"]]
        print(f"    Feature type list: {fL}")

### 2. Fetch just the "binding site" features for PDB instance 3V85.A
Now, we'll do the same query, but apply a filter to only return the `BINDING_SITE` features (<a href="https://sequence-coordinates.rcsb.org/graphiql/index.html?query=%7B%0A%20%20annotations(%0A%20%20%20%20filters%3A%20%5B%7Bvalues%3A%20%5B%22BINDING_SITE%22%5D%2C%20operation%3A%20EQUALS%2C%20field%3A%20TYPE%7D%5D%0A%20%20%20%20queryId%3A%20%223V85.A%22%0A%20%20%20%20reference%3A%20PDB_INSTANCE%0A%20%20%20%20sources%3A%20%5BUNIPROT%2C%20PDB_ENTITY%2C%20PDB_INSTANCE%5D%0A%20%20)%20%7B%0A%20%20%20%20target_id%0A%20%20%20%20source%0A%20%20%20%20features%20%7B%0A%20%20%20%20%20%20provenance_source%0A%20%20%20%20%20%20description%0A%20%20%20%20%20%20type%0A%20%20%20%20%20%20additional_properties%20%7B%0A%20%20%20%20%20%20%20%20property_name%0A%20%20%20%20%20%20%20%20property_value%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%20%20feature_positions%20%7B%0A%20%20%20%20%20%20%20%20open_begin%0A%20%20%20%20%20%20%20%20values%0A%20%20%20%20%20%20%20%20beg_seq_id%0A%20%20%20%20%20%20%20%20value%0A%20%20%20%20%20%20%20%20open_end%0A%20%20%20%20%20%20%20%20range_id%0A%20%20%20%20%20%20%20%20end_ori_id%0A%20%20%20%20%20%20%20%20end_seq_id%0A%20%20%20%20%20%20%20%20beg_ori_id%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%20%20value%0A%20%20%20%20%20%20feature_id%0A%20%20%20%20%20%20name%0A%20%20%20%20%7D%0A%20%20%7D%0A%7D">API Query Editor link</a>):

In [ ]:
from rcsbapi.sequence import Annotations, AnnotationFilterInput
from rich import print as rprint

# Fetch only binding_site features for a particular PDB Instance
query = Annotations(
    query_id="3V85.A",
    reference="PDB_INSTANCE",
    sources=["UNIPROT", "PDB_ENTITY", "PDB_INSTANCE"],
    return_data_list=["target_id", "source", "features"],
    filters=[
        AnnotationFilterInput(
            field="TYPE",
            operation="EQUALS",
            values=["BINDING_SITE"],
        )
    ],
)
result_dict = query.exec()
print(f"API Query Editor link: {query.get_editor_link()}")
rprint(result_dict)

### 3. Fetch the binding site features for _all_ PDB instances of UniProt Q9SIY3
(<a href="https://sequence-coordinates.rcsb.org/graphiql/index.html?query=%7B%0A%20%20annotations(%0A%20%20%20%20filters%3A%20%5B%7Bvalues%3A%20%5B%22BINDING_SITE%22%5D%2C%20operation%3A%20EQUALS%2C%20field%3A%20TYPE%7D%5D%0A%20%20%20%20queryId%3A%20%22Q9SIY3%22%0A%20%20%20%20reference%3A%20UNIPROT%0A%20%20%20%20sources%3A%20%5BPDB_INSTANCE%5D%0A%20%20)%20%7B%0A%20%20%20%20target_id%0A%20%20%20%20source%0A%20%20%20%20features%20%7B%0A%20%20%20%20%20%20feature_id%0A%20%20%20%20%20%20feature_positions%20%7B%0A%20%20%20%20%20%20%20%20beg_ori_id%0A%20%20%20%20%20%20%20%20beg_seq_id%0A%20%20%20%20%20%20%7D%0A%20%20%20%20%20%20name%0A%20%20%20%20%20%20type%0A%20%20%20%20%20%20description%0A%20%20%20%20%20%20provenance_source%0A%20%20%20%20%7D%0A%20%20%7D%0A%7D">API Query Editor link</a>)

In [ ]:
from rcsbapi.sequence import Annotations, AnnotationFilterInput

query = Annotations(
    query_id="Q9SIY3",
    reference="UNIPROT",
    sources=["PDB_INSTANCE"],  # Fetch only binding site annotations from the PDB_INSTANCE data source
    # Only return relevant data items for binding sites
    return_data_list=[
        "target_id",
        "source",
        "features.feature_id",
        "features.feature_positions.beg_ori_id",
        "features.feature_positions.beg_seq_id",
        "features.name",
        "features.type",
        "features.description",
        "features.provenance_source",
    ],
    filters=[
        AnnotationFilterInput(
            field="TYPE",
            operation="EQUALS",
            values=["BINDING_SITE"],
        )
    ],
)
result_dict = query.exec()
print("Number of annotations: ", len(result_dict["data"]["annotations"]))
print(f"API Query Editor link: {query.get_editor_link()}")

## GraphiQL Query Editor
As demonstrated in the examples above, one helpful feature available is the ability to view and test your query in our [Sequence Coordinates API GraphiQL query editor](https://sequence-coordinates.rcsb.org/graphiql/index.html), by using the `.get_editor_link()` method, e.g.:

In [ ]:
from rcsbapi.sequence import Alignments

query = Alignments(
    query_id="Q9SIY3",
    db_from="UNIPROT",
    db_to="PDB_ENTITY",
    return_data_list=["query_sequence", "target_alignments", "alignment_length"]
)
print(f"API Query Editor link: {query.get_editor_link()}")

## Further Documentation
For more extensive examples and implementation details visit our [readthedocs](https://rcsbapi.readthedocs.io/en/latest/seq_api/quickstart.html).